# Week 1: T4 + vLLM smoke test

Target: **Kaggle Notebooks** with a **T4 x1** or **T4 x2** GPU runtime.

Goal: prove the validation path works end-to-end on free-tier hardware.
- vLLM installs cleanly on T4
- TinyLlama-1.1B-Chat loads and generates
- A single decode-throughput number is recorded and written to `smoke_test_result.json`
- The resulting JSON is what Week 2's full sweep (`03_vllm_t4_validation.ipynb`) will extend

This notebook does **not** yet produce the predicted-vs-measured chart. That lands in Week 2 together with the full sweep across batch sizes and input lengths.

Expected decode TPS (from `core.cost_engine.decode_tps`): ~**60-120 tok/s** at batch=1, FP16, memory-bound. Measured numbers in this ballpark pass the smoke test.

## 1. Install dependencies

Kaggle base image has PyTorch and CUDA already. vLLM wheel install takes 1-2 minutes.

In [ ]:
!pip install -q 'vllm==0.6.3' 2>&1 | tail -n 3

In [ ]:
import json, time, subprocess, platform, torch

env = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'gpu_count': torch.cuda.device_count(),
}
print(json.dumps(env, indent=2))
assert env['cuda_available'], 'No CUDA GPU attached. Switch runtime to T4 x1.'
assert 'T4' in env['gpu_name'], f"Expected T4, got {env['gpu_name']}. This notebook is specifically for T4 validation."

## 2. Load TinyLlama-1.1B

Pin the model revision so the measured TPS is reproducible.

In [ ]:
from vllm import LLM, SamplingParams

MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
REVISION = 'main'  # pin to a commit for reproducibility in week 2

llm = LLM(
    model=MODEL,
    revision=REVISION,
    dtype='float16',
    gpu_memory_utilization=0.85,
    max_model_len=2048,
    enforce_eager=True,   # avoid cuda graph for a first run
)

## 3. Decode throughput, batch=1

Generate 128 tokens from a short prompt and record elapsed time.

In [ ]:
PROMPT = 'Write a short paragraph about the economics of GPU clusters.'
MAX_NEW = 128

params = SamplingParams(temperature=0.0, max_tokens=MAX_NEW, ignore_eos=True)

# warm-up
_ = llm.generate([PROMPT], params)

t0 = time.perf_counter()
out = llm.generate([PROMPT], params)
t1 = time.perf_counter()

gen_tokens = len(out[0].outputs[0].token_ids)
elapsed = t1 - t0
decode_tps = gen_tokens / elapsed

print(f'generated {gen_tokens} tokens in {elapsed:.2f}s => {decode_tps:.1f} tok/s')

## 4. Write result for commit back to repo

The Streamlit validation page reads from `data/processed/t4_smoke_test.json`. Download this file from the Kaggle notebook outputs and commit it to the repo.

In [ ]:
from datetime import datetime, timezone

result = {
    'kind': 'measured',
    'notebook': '00_t4_smoke_test',
    'retrieved_at': datetime.now(timezone.utc).isoformat(),
    'gpu': 't4',
    'model': MODEL,
    'revision': REVISION,
    'dtype': 'float16',
    'engine': 'vllm',
    'batch_size': 1,
    'input_length': None,
    'output_length': gen_tokens,
    'elapsed_seconds': elapsed,
    'decode_tokens_per_second': decode_tps,
    'env': env,
    'notes': 'single-prompt smoke test; warm-up run discarded; FP16; enforce_eager=True',
}

with open('smoke_test_result.json', 'w') as f:
    json.dump(result, f, indent=2)

print(json.dumps(result, indent=2))

## What next

1. Download `smoke_test_result.json` from this Kaggle notebook's outputs.
2. Place it at `data/processed/t4_smoke_test.json` in the repo and commit.
3. Week 2 replaces this notebook with `03_vllm_t4_validation.ipynb`, which sweeps models (1B / 1.5B / 3B), batch sizes (1, 4, 16, 32), and input lengths (128, 512, 2048), writing a `t4_validation.parquet` that the Streamlit validation page plots against predicted TPS.